In [1]:
!pip install uv
!uv pip install lambeq pandas tqdm pennylane torch clip mlflow cotengra optuna
!pip install git+https://github.com/openai/CLIP.git

Using Python 3.12.12 environment at: /usr
Audited 9 packages in 100ms
  Cloning https://github.com/openai/CLIP.git to /tmp/pip-req-build-_8yxv510
  Running command git clone --filter=blob:none --quiet https://github.com/openai/CLIP.git /tmp/pip-req-build-_8yxv510
  Resolved https://github.com/openai/CLIP.git to commit ded190a052fdf4585bd685cee5bc96e0310d2c93
  Preparing metadata (setup.py) ... done


In [2]:
from torch.utils.data import DataLoader
import matplotlib.pyplot as plt
from tqdm import trange, tqdm
import pandas as pd
import numpy as np
import torch, os, clip, pickle, mlflow, time, sys
from google.colab import drive

root_path = os.path.join(os.getcwd(),'drive/MyDrive/tls_qclip')

In [3]:
drive.mount('/content/drive')
os.chdir(root_path)

from modules.data_processing import *
from modules.model import *
from modules.util import *

txt_path = os.path.join(root_path, 'ARO/text_circuits/attribution')
img_path = os.path.join(root_path, 'ARO/images')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [4]:
OBMAP = (5,5,5)
NLAYERS = 3
train_einsum = load_pkl(os.path.join(txt_path, 'aro_train_einsum_as_53.pkl'))
valid_einsum = load_pkl(os.path.join(txt_path, 'aro_valid_einsum_as_53.pkl'))
test_einsum = load_pkl(os.path.join(txt_path, 'aro_test_einsum_as_53.pkl'))

IMG_DIM = 512
train_img = load_pkl(os.path.join(img_path, 'att_imgenc_train_512.pkl'))
valid_img = load_pkl(os.path.join(img_path, 'att_imgenc_valid_512.pkl'))
test_img = load_pkl(os.path.join(img_path, 'att_imgenc_test_512.pkl'))

In [5]:
train_pos_einsum, train_neg_einsum = zip(*train_einsum)
valid_pos_einsum, valid_neg_einsum = zip(*valid_einsum)
test_pos_einsum, test_neg_einsum = zip(*test_einsum)

In [6]:
arr2type = lambda arr, type: [ele.to(type) for ele in arr]
train_img = arr2type(train_img, torch.complex128)
valid_img = arr2type(valid_img, torch.complex128)
test_img = arr2type(test_img, torch.complex128)

In [7]:
print(max_width(train_pos_einsum[0][0], train_pos_einsum[0][1]))

8192


In [8]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
torch.set_default_device(device)
torch.set_default_dtype(torch.float64)

model = EinsumModel()
model.precision = torch.complex128
model.from_einsum(train_pos_einsum + train_neg_einsum + valid_pos_einsum + valid_neg_einsum + test_pos_einsum + test_neg_einsum)

model.initialise_weights(near_id=False, temptrain=False)
model.to(device)
TPARAMS = sum(p.numel() for p in model.parameters() if p.requires_grad)

if next(model.parameters()).is_cuda and torch.cuda.is_available():
    print("Model is on GPU")
else:
    print("Running on CPU...")
print(f"Parameters: #{TPARAMS}")

100%|██████████| 57474/57474 [00:04<00:00, 14309.23it/s]


Running on CPU...
Parameters: #92889


In [9]:
BATCH_SIZE = 256
N_EXAMPLES = len(train_einsum)

generator = torch.Generator(device=device)
train_dataset = CLIP_Dataset(train_pos_einsum, train_neg_einsum, train_img)
train_dataset = subsample_data(train_dataset)
train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=False, collate_fn=collate_fn, generator=generator)

valid_dataset = CLIP_Dataset(valid_pos_einsum, valid_neg_einsum, valid_img)
valid_loader = DataLoader(valid_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, generator=generator)

test_dataset = CLIP_Dataset(test_pos_einsum, test_neg_einsum, test_img)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE, collate_fn=collate_fn, generator=generator)

model.compile_dataset(train_loader, 'aro')
model.compile_dataset(valid_loader, 'aro')
model.compile_dataset(test_loader, 'aro')
print(f"Unique Contractions: {len(model.path_cache)}")

Unique Contractions: 141


In [10]:
dtime = time.strftime("%m_%d_%H:%M:%S")
fpath = os.path.join(root_path, 'aro_runs', dtime)
if not os.path.exists(fpath):
    os.makedirs(fpath)
db_path = os.path.join(root_path, 'mlf.db')
mlflow.pytorch.autolog()
mlflow.set_tracking_uri(f"sqlite:///{db_path}")
mlflow.set_experiment('qclip_aro_att')

<Experiment: artifact_location='/content/drive/MyDrive/tls_qclip/mlruns/4', creation_time=1772638391935, experiment_id='4', last_update_time=1772638391935, lifecycle_stage='active', name='qclip_aro_att', tags={}, workspace='default'>

In [11]:
EPOCHS = 100
LEARNING_RATE = 1e-2
LR_FACTOR = 0.5
MPATIENCE = 3
SEED = int.from_bytes(os.urandom(4))
set_seed(SEED)

optimizer = torch.optim.AdamW([{'params': model.weights, 'lr': LEARNING_RATE}], weight_decay=1e-5)
# {'params': model.word_importance, 'lr': LEARNING_RATE/10}, {'params': model.log_temp, 'lr': 0.01}], weight_decay=1e-5)
scheduler = torch.optim.lr_scheduler.ReduceLROnPlateau(optimizer, mode='max', factor=LR_FACTOR, patience=5)
loss_fn = QInfoNCE_cos(device=device)
acc_fn = fs_distance

In [ ]:
trajectory = []
initial_params = torch.cat([p.view(-1) for p in model.parameters()]).detach().clone()
print(f"Starting training at: {dtime}")

with mlflow.start_run(run_name=dtime):
    # --- Log Hyperparameters ---
    mlflow.log_param("seed", SEED)
    mlflow.log_param("learning_rate_factor", LR_FACTOR)
    mlflow.log_param("patience", MPATIENCE)
    mlflow.log_param("learning_rate", LEARNING_RATE)
    mlflow.log_param("batch_size", BATCH_SIZE)
    mlflow.log_param("total_parameters", TPARAMS)
    mlflow.log_param("type_qubits", OBMAP)
    mlflow.log_param("ansatze_layers", NLAYERS)


    # --- Start Training ---
    for epoch in trange(EPOCHS):
        # --- Training Pass ---
        train_start = time.time()
        train_loss, train_acc, avg_grad_norm = update_model_aro(model, train_loader, loss_fn, acc_fn, optimizer)
        train_end = time.time()

        # --- Validation Pass ---
        valid_start = time.time()
        valid_acc = eval_model_aro(model, valid_loader, acc_fn)
        valid_end = time.time()

        # --- Record Time & Save Current Weights ---
        model.save(os.path.join(fpath, 'model.lt'))
        train_time = train_end - train_start
        valid_time = valid_end - valid_start

        # --- Compute & Log Relevant Metrics ---
        current_lr = optimizer.param_groups[0]["lr"]
        current_w = torch.nn.utils.parameters_to_vector(model.parameters()).detach().cpu()
        trajectory.append(current_w)
        init_mad = torch.mean(torch.abs(current_w - initial_params)).item()
        theta_var = torch.mean(torch.abs(current_w - torch.mean(current_w))).item()

        mlflow.log_metric("train_loss", train_loss, step=epoch+1)
        mlflow.log_metric("train_accuracy", train_acc, step=epoch+1)
        mlflow.log_metric("valid_accuracy", valid_acc, step=epoch+1)
        mlflow.log_metric("avg_grad_norm", avg_grad_norm, step=epoch+1)
        mlflow.log_metric("epoch_time", train_time + valid_time, step=epoch+1)
        mlflow.log_metric("learning_rate", current_lr, step=epoch+1)
        mlflow.log_metric("temperature", model.temp, step=epoch+1)
        mlflow.log_metric("parameter_drift", init_mad, step=epoch+1)
        mlflow.log_metric("parameter_variance", theta_var, step=epoch+1)

        # --- Print Current Epoch Information ---
        tqdm.write(f"Epoch {epoch+1}: Loss {train_loss:.4f}, Train Acc {train_acc:.4f}, Val Acc {valid_acc:.4f}, Avg Grad Norm {avg_grad_norm:.2e}, Train Time {train_time/60:.2f}, Valid Time {valid_time/60:.2f} (LR: {current_lr}, T: {model.temp.item():.5f}), Drift: {init_mad:.4f}, Variance: {theta_var:.4f}".strip(), file=sys.stderr)

        if (current_lr <= LEARNING_RATE*LR_FACTOR**MPATIENCE) or (epoch + 1 == EPOCHS):
            mlflow.log_param("epochs", epoch + 1)
            break
        else:
            scheduler.step(valid_acc)

    # --- Evaluate Model on Test Set ---
    test_acc = eval_model_aro(model, test_loader, acc_fn)
    mlflow.log_metric("test_accuracy", test_acc, step=epoch+1)
    print(f"Test Accuracy: {test_acc}")

store_pkl(trajectory, os.path.join(fpath,'trj.pkl'))
mlflow.end_run()

Starting training at: 03_04_15:36:37


Epoch 1: Loss 5.5311, Train Acc 0.6472, Val Acc 0.6091, Avg Grad Norm 2.98e-01, Train Time 9.96, Valid Time 0.92 (LR: 0.01, T: 0.07000), Drift: 0.0308, Variance: 0.3939
Epoch 2: Loss 5.4112, Train Acc 0.7790, Val Acc 0.6542, Avg Grad Norm 3.21e-01, Train Time 9.76, Valid Time 0.91 (LR: 0.01, T: 0.07000), Drift: 0.0417, Variance: 0.3950
Epoch 3: Loss 5.3209, Train Acc 0.8384, Val Acc 0.6883, Avg Grad Norm 3.17e-01, Train Time 9.95, Valid Time 0.87 (LR: 0.01, T: 0.07000), Drift: 0.0483, Variance: 0.3956
Epoch 4: Loss 5.2603, Train Acc 0.8609, Val Acc 0.6932, Avg Grad Norm 3.22e-01, Train Time 9.77, Valid Time 0.87 (LR: 0.01, T: 0.07000), Drift: 0.0534, Variance: 0.3962
Epoch 5: Loss 5.2184, Train Acc 0.8718, Val Acc 0.7040, Avg Grad Norm 3.22e-01, Train Time 9.79, Valid Time 0.86 (LR: 0.01, T: 0.07000), Drift: 0.0574, Variance: 0.3968
Epoch 6: Loss 5.1906, Train Acc 0.8811, Val Acc 0.6985, Avg Grad Norm 3.26e-01, Train Time 9.64, Valid Time 0.86 (LR: 0.01, T: 0.07000), Drift: 0.0609, Var

Test Accuracy: 0.6872527051033592
